# Scenario Construction

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import BenignFlowExtractor, WindowDatasetBuilder, WindowScenarioBuilder

## 1. Create Dataset Similarity

### 1.1 Extract Benign Flows

In [ ]:
extractBenignFlows = False
benignExtractionDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
benignFlowsOutputPath = Path("data/benign/benign_flows.csv")
benignExtractionChunkSize = 100_000
benignSourceFiles = None
overwriteBenignFlows = False

if extractBenignFlows:
    benignExtraction = BenignFlowExtractor(
        dataDirectory=benignExtractionDirectory,
        chunkSize=benignExtractionChunkSize,
    ).extract(
        outputPath=benignFlowsOutputPath,
        sourceFiles=benignSourceFiles,
        overwrite=overwriteBenignFlows,
    )
    display(pd.DataFrame([benignExtraction]))


### 1.2 Aggregate Existing Benign Flows

In [ ]:
benignFlowsPath = Path("data/benign/benign_flows.csv")
benignAggregationOutputDirectory = Path("data")
benignAggregationChunkSize = 100_000
benignWindowSeconds = [1.0]

benignDatasets = WindowDatasetBuilder(
    dataDirectory=benignFlowsPath.parent,
    outputDirectory=benignAggregationOutputDirectory,
    chunkSize=benignAggregationChunkSize,
).aggregateBenignFlows(
    benignFlowsPath=benignFlowsPath,
    windowSeconds=benignWindowSeconds,
)

display(benignDatasets["summary"])


### 1.3 Aggregate Mixed Attack Flows

In [ ]:
attackDataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
attackAggregationOutputDirectory = Path("data")
attackAggregationChunkSize = 100_000
attackWindowSeconds = [1.0]
cutMinutes = None
cutStartMinutes = {}

attackFiles = {'NTP': 'DrDoS_NTP.csv',
    'DNS': 'DrDoS_DNS.csv',
    'LDAP': 'DrDoS_LDAP.csv',
    'MSSQL': 'DrDoS_MSSQL.csv',
    'NETBIOS': 'DrDoS_NetBIOS.csv',
    'SNMP': 'DrDoS_SNMP.csv',
    'SSDP': 'DrDoS_SSDP.csv',
    'UDP': 'DrDoS_UDP.csv',
    'UDP-LAG': 'UDPLag.csv',
    'SYN': 'Syn.csv',
    'TFTP': 'TFTP.csv'}

attackDatasets = WindowDatasetBuilder(
    dataDirectory=attackDataDirectory,
    outputDirectory=attackAggregationOutputDirectory,
    chunkSize=attackAggregationChunkSize,
).aggregateAttackFlows(
    attackFiles=attackFiles,
    windowSeconds=attackWindowSeconds,
    cutMinutes=cutMinutes,
    cutStartMinutes=cutStartMinutes,
)

display(attackDatasets["summary"])


### 1.4 Concatenate Similarity Dataset

In [4]:
similarityOutputPath = Path("data/similarity/similarity_1s.csv")
similarityBenignPath = Path("data/benign/benign_1s.csv")
similarityAttacksDirectory = Path("data/attacks/1s")
similaritySequence = ["NTP", "DNS", "LDAP", "MSSQL", "NETBIOS", "SNMP", "SSDP", "UDP", "UDP-LAG", "SYN", "TFTP"]
similarityBlockSeconds = 180
similarityInitialBenignSeconds = 2000
similarityBetweenBenignSeconds = 600
similaritySamplingSeed = 42
similarityAttackPositionsSeconds = None
similarityBenignStartSeconds = 0

similarityDataset = WindowScenarioBuilder(
    benignPath=similarityBenignPath,
    attacksDirectory=similarityAttacksDirectory,
    windowSeconds=1.0,
).build(
    sequence=similaritySequence,
    outputPath=similarityOutputPath,
    blockSeconds=similarityBlockSeconds,
    initialBenignSeconds=similarityInitialBenignSeconds,
    betweenBenignSeconds=similarityBetweenBenignSeconds,
    randomSeed=similaritySamplingSeed,
    attackPositionsSeconds=similarityAttackPositionsSeconds,
    benignStartSeconds=similarityBenignStartSeconds,
    allowBenignReuse=True,
)

display(similarityDataset["summary"])


,BlockIndex,ScenarioPart,TargetAttack,Windows,Start,End,AttackWindows,BeginningWindows,MiddleWindows,EndWindows,OccupiedSeconds
0,0,benign,BENIGN,2000,2021-01-01 00:00:00,2021-01-01 00:33:19,0,0,0,0,2000.0
1,0,mixed_attack,NTP,180,2021-01-01 00:33:20,2021-01-01 00:36:19,99,60,60,60,180.0
2,1,benign,BENIGN,600,2021-01-01 00:36:20,2021-01-01 00:46:19,0,0,0,0,600.0
3,1,mixed_attack,DNS,180,2021-01-01 00:46:20,2021-01-01 00:49:19,170,60,60,60,180.0
4,2,benign,BENIGN,600,2021-01-01 00:49:20,2021-01-01 00:59:19,0,0,0,0,600.0
5,2,mixed_attack,LDAP,180,2021-01-01 00:59:20,2021-01-01 01:02:19,180,60,60,60,180.0
6,3,benign,BENIGN,600,2021-01-01 01:02:20,2021-01-01 01:12:19,0,0,0,0,600.0
7,3,mixed_attack,MSSQL,180,2021-01-01 01:12:20,2021-01-01 01:15:19,155,60,60,60,180.0
8,4,benign,BENIGN,600,2021-01-01 01:15:20,2021-01-01 01:25:19,0,0,0,0,600.0
9,4,mixed_attack,NETBIOS,180,2021-01-01 01:25:20,2021-01-01 01:28:19,176,60,60,60,180.0


## 2. Create Controlled Scenarios

In [ ]:
scenarioWindowSeconds = [1.0]
scenarioOutputDirectory = Path("data/scenarios")
scenarioSequences = {
    "Consistency": ["DNS", "DNS", "DNS"],
    "Generalization": ["DNS", "LDAP", "DNS"],
    "Adaptation": ["DNS", "SYN", "DNS"],
    "Recurrence": ["DNS", "SYN", "LDAP", "DNS", "SYN", "LDAP"],
}
scenarioBlockSeconds = {name: 60 for name in scenarioSequences}
scenarioInitialBenignSeconds = 2000
scenarioBetweenBenignSeconds = 120
scenarioSamplingSeed = 42
scenarioAttackPositionsSeconds = {}
scenarioBenignStartSeconds = 0

scenarioResults = []
for seconds in scenarioWindowSeconds:
    durationName = f"{seconds:g}s"
    builder = WindowScenarioBuilder(
        benignPath=Path(f"data/benign/benign_{durationName}.csv"),
        attacksDirectory=Path(f"data/attacks/{durationName}"),
        windowSeconds=seconds,
    )
    for name, sequence in scenarioSequences.items():
        result = builder.build(
            sequence=sequence,
            outputPath=scenarioOutputDirectory / durationName / f"{name}_{durationName}.csv",
            blockSeconds=scenarioBlockSeconds[name],
            initialBenignSeconds=scenarioInitialBenignSeconds,
            betweenBenignSeconds=scenarioBetweenBenignSeconds,
            randomSeed=scenarioSamplingSeed,
            attackPositionsSeconds=scenarioAttackPositionsSeconds.get(name),
            benignStartSeconds=scenarioBenignStartSeconds,
            allowBenignReuse=True,
        )
        scenarioResults.append({"Scenario": name, "WindowSeconds": seconds, "Windows": result["windows"],
                                "AttackWindows": result["attackWindows"], "Path": result["datasetPath"]})

display(pd.DataFrame(scenarioResults))
